In [1]:
from python_speech_features import mfcc

import matplotlib.pyplot as plt
from scipy.io import wavfile
import argparse
import os
from glob import glob
import numpy as np
import pandas as pd
from librosa.core import resample, to_mono
from tqdm import tqdm
import wavio

In [27]:
def envelope(y, rate, threshold):
    mask = []
    y = pd.Series(y).apply(np.abs)
    y_mean = y.rolling(window=int(rate/10),
                       min_periods=1,
                       center=True).max()
    for mean in y_mean:
        if mean > threshold:
            mask.append(True)
        else:
            mask.append(False)
    return mask, y_mean


def downsample_mono(path, sr):
    obj = wavio.read(path)
    wav = obj.data.astype(np.float32, order='F')
    rate = obj.rate
    try:
        channel = wav.shape[1]
        if channel == 2:
            wav = to_mono(wav.T)
        elif channel == 1:
            wav = to_mono(wav.reshape(-1))
    except IndexError:
        wav = to_mono(wav.reshape(-1))
        pass
    except Exception as exc:
        raise exc
    wav = resample(wav, rate, sr)
    wav = wav.astype(np.int16)
    return sr, wav


def save_sample(sample, rate, target_dir, fn, ix):
    fn = fn.split('.wav')[0]
    dst_path = os.path.join(target_dir.split('.')[0], fn+'_{}.wav'.format(str(ix)))
    if os.path.exists(dst_path):
        return
    wavfile.write(dst_path, rate, sample)


def check_dir(path):
    if os.path.exists(path) is False:
        os.mkdir(path)


def split_wavs(args):
    src_root = args.src_root
    dst_root = args.dst_root
    dt = args.delta_time

    wav_paths = glob('{}/**'.format(src_root), recursive=True)
    wav_paths = [x for x in wav_paths if '.wav' in x]
    dirs = os.listdir(src_root)
    check_dir(dst_root)
    classes = os.listdir(src_root)
    for _cls in classes:
        target_dir = os.path.join(dst_root, _cls)
        check_dir(target_dir)
        src_dir = os.path.join(src_root, _cls)
        for fn in tqdm(os.listdir(src_dir)):
            src_fn = os.path.join(src_dir, fn)
            rate, wav = downsample_mono(src_fn, args.sr)
            mask, y_mean = envelope(wav, rate, threshold=args.threshold)
            wav = wav[mask]
            delta_sample = int(dt*rate)

            # cleaned audio is less than a single sample
            # pad with zeros to delta_sample size
            if wav.shape[0] < delta_sample:
                sample = np.zeros(shape=(delta_sample,), dtype=np.int16)
                sample[:wav.shape[0]] = wav
                save_sample(sample, rate, target_dir, fn, 0)
            # step through audio and save every delta_sample
            # discard the ending audio if it is too short
            else:
                trunc = wav.shape[0] % delta_sample
                for cnt, i in enumerate(np.arange(0, wav.shape[0]-trunc, delta_sample)):
                    start = int(i)
                    stop = int(i + delta_sample)
                    sample = wav[start:stop]
                    save_sample(sample, rate, target_dir, fn, cnt)


def test_threshold(args):
    src_root = args.src_root
    wav_paths = glob('{}/**'.format(src_root), recursive=True)
    wav_path = [x for x in wav_paths if args.fn in x]
    if len(wav_path) != 1:
        print('audio file not found for sub-string: {}'.format(args.fn))
        return
    rate, wav = downsample_mono(wav_path[0], args.sr)
    mask, env = envelope(wav, rate, threshold=args.threshold)
    plt.style.use('ggplot')
    plt.title('Signal Envelope, Threshold = {}'.format(str(args.threshold)))
    plt.plot(wav[np.logical_not(mask)], color='r', label='remove')
    plt.plot(wav[mask], color='c', label='keep')
    plt.plot(env, color='m', label='envelope')
    plt.grid(False)
    plt.legend(loc='best')
    plt.show()

In [12]:
import pandas as pd

from glob import glob
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
%matplotlib inline

In [10]:
data_dir = "../../../RAVDESS"
save_dir = "../../dataset/"
clean_dir = save_dir+"clean/"

data_actors = os.listdir(data_dir)

# emotions = os.listdir(save_dir)
emo_label = {"neutral":0,"calm":1,"happy":2,"sad":3,"angry":4,"fearful":5,"disgust":6,"surprised":7}
int_label = {"01":"normal","02":"strong"}
stmt_label = {"01":"Kids are talking by the door", "02":"Dogs are sitting by the door"}
sex_label = {"male":1,"female":0}
emotions = {"01":"neutral","02":"calm","03":"happy","04":"sad","05":"angry","06":"fearful","07":"disgust","08":"surprised"}

## CLEAN DATA

In [73]:
audio_df = {"filename":[],"sr":[],"cleaned_sr":[],"length":[],"cleaned_length":[],"emotion":[],"emo_label":[],"intensity":[],"int_label":[],"stmt":[],"stmt_label":[],"repetition":[],"sex":[],"sex_label":[],"actor":[]}
for actor in data_actors:
    files = glob(data_dir+"/"+actor+"/*.wav")
    for file in files:
        direc, filename = file.split("\\")
        savename = filename[6:]
        
        filename = savename.replace(".wav","")[3:]
        
        intensity = filename[:2]
        intens_label = int_label[intensity]
        
        stmt = filename[3:5]
        stmt_lab = stmt_label[stmt]
        
        actor = filename[-2:]
        repetition = filename[-5:-3]
        
        sex = "male" if int(actor)%2 == 1 else "female"
        sex_lab = sex_label[sex]
        
        
        emo_lab = savename[:2]
        emotion = emotions[emo_lab]
        emo_val = emo_label[emotion]
        
        emo_dir = clean_dir + emotion+"/"
        
        signal,sr = librosa.load(file,sr=None)
        length = signal.shape[0]/sr
        
        c_signal,c_sr = librosa.load(file,sr=16000)
        mask = envelope(c_signal,c_sr,0.0005)
        c_signal = c_signal[mask[0]]
        cleaned_length = c_signal.shape[0]/c_sr
        
        
#         mel = mfcc(signal[:c_sr],c_sr,numcep=13,nfilt=26,nfft=1200).T
        
        audio_df["filename"].append(savename)
        audio_df["sr"].append(sr)
        audio_df["cleaned_sr"].append(c_sr)
        audio_df["length"].append(length)
        audio_df["cleaned_length"].append(cleaned_length)
        audio_df["emotion"].append(emotion)
        audio_df["emo_label"].append(emo_val)
        audio_df["intensity"].append(intensity)
        audio_df["int_label"].append(intens_label)
        audio_df["stmt"].append(stmt)
        audio_df["stmt_label"].append(stmt_lab)
        audio_df["repetition"].append(repetition)
        audio_df["sex"].append(sex)
        audio_df["sex_label"].append(sex_lab)
        audio_df["actor"].append(actor)
        
        if not os.path.exists(clean_dir):
            os.makedirs(clean_dir)
            
        wavfile.write(f"{clean_dir}{savename}",c_sr,c_signal)

In [51]:
wavfile.write("xd2.wav",sr,signal2)

In [42]:
length

1.837625

In [72]:
pd.DataFrame(audio_df).to_csv("audio_data.csv")
pd.DataFrame(audio_df)

,filename,sr,cleaned_sr,length,cleaned_length,emotion,emo_label,intensity,int_label,stmt,stmt_label,repetition,sex,sex_label,actor
0,01-01-01-01-01.wav,48000,16000,3.303292,1.515937,neutral,0,01,normal,01,Kids are talking by the door,01,male,1,01
1,01-01-01-02-01.wav,48000,16000,3.336667,1.649563,neutral,0,01,normal,01,Kids are talking by the door,02,male,1,01
2,01-01-02-01-01.wav,48000,16000,3.269917,1.419875,neutral,0,01,normal,02,Dogs are sitting by the door,01,male,1,01
3,01-01-02-02-01.wav,48000,16000,3.169833,1.351438,neutral,0,01,normal,02,Dogs are sitting by the door,02,male,1,01
4,02-01-01-01-01.wav,48000,16000,3.536854,1.714063,calm,1,01,normal,01,Kids are talking by the door,01,male,1,01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,08-01-02-02-24.wav,48000,16000,3.403396,1.873125,surprised,7,01,normal,02,Dogs are sitting by the door,02,female,0,24
1436,08-02-01-01-24.wav,48000,16000,3.937271,2.400875,surprised,7,02,strong,01,Kids are talking by the door,01,female,0,24
1437,08-02-01-02-24.wav,48000,16000,3.970625,2.394687,surprised,7,02,strong,01,Kids are talking by the door,02,female,0,24
1438,08-02-02-01-24.wav,48000,16000,3.670333,2.489875,surprised,7,02,strong,02,Dogs are sitting by the door,01,female,0,24


In [70]:
pd.read_csv("audio_data.csv",index_col=[0])

,filename,sr,length,cleaned_length,emotion,emo_label,intensity,int_label,stmt,stmt_label,repetition,sex,sex_label,actor
0,01-01-01-01-01.wav,48000,3.303292,1.571937,neutral,0,1,normal,1,Kids are talking by the door,1,male,1,1
1,01-01-01-02-01.wav,48000,3.336667,1.666083,neutral,0,1,normal,1,Kids are talking by the door,2,male,1,1
2,01-01-02-01-01.wav,48000,3.269917,1.419917,neutral,0,1,normal,2,Dogs are sitting by the door,1,male,1,1
3,01-01-02-02-01.wav,48000,3.169833,1.351521,neutral,0,1,normal,2,Dogs are sitting by the door,2,male,1,1
4,02-01-01-01-01.wav,48000,3.536854,1.814375,calm,1,1,normal,1,Kids are talking by the door,1,male,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435,08-01-02-02-24.wav,48000,3.403396,1.880688,surprised,7,1,normal,2,Dogs are sitting by the door,2,female,0,24
1436,08-02-01-01-24.wav,48000,3.937271,2.420396,surprised,7,2,strong,1,Kids are talking by the door,1,female,0,24
1437,08-02-01-02-24.wav,48000,3.970625,2.415563,surprised,7,2,strong,1,Kids are talking by the door,2,female,0,24
1438,08-02-02-01-24.wav,48000,3.670333,2.491146,surprised,7,2,strong,2,Dogs are sitting by the door,1,female,0,24


In [55]:
pd.DataFrame(audio_df)

ValueError: All arrays must be of the same length